In [1]:
DATASET = "nyc_tlc_yellow"
BRONZE_PATH = f"../lakehouse/bronze/{DATASET}"
SILVER_PATH = f"../lakehouse/silver/{DATASET}"

RAW_FILE = "yellow_tripdata_2024-01.parquet"  # only for metadata/logging if needed
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)

BRONZE_PATH: ../lakehouse/bronze/nyc_tlc_yellow
SILVER_PATH: ../lakehouse/silver/nyc_tlc_yellow


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC-TLC-Lakehouse")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .getOrCreate()
)

spark

26/02/23 20:57:05 WARN Utils: Your hostname, willian-A320M-S2H resolves to a loopback address: 127.0.1.1; using 192.168.0.110 instead (on interface enp6s0)
26/02/23 20:57:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/willian/PhaifferTech/nyc-tlc-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/willian/.ivy2/cache
The jars for the packages stored in: /home/willian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3e505202-7cb0-441f-828a-9f692d9c451c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 180ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   

In [3]:
df_bronze = spark.read.format("delta").load(BRONZE_PATH)

print("bronze_rows:", df_bronze.count())
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

26/02/23 20:57:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


bronze_rows: 2964624
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _

26/02/23 20:57:21 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------------+-------------------------------+--------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|_ingestion_timestamp      |_source_file                   |_dataset      |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------------+-------------

In [4]:
required_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

missing = [c for c in required_cols if c not in df_bronze.columns]
if missing:
    raise ValueError(f"Missing required columns in Bronze: {missing}")

print("All required columns exist.")

All required columns exist.


In [5]:
from pyspark.sql.functions import col, to_date

df_silver = (
    df_bronze
    .filter(col("tpep_pickup_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime") >= col("tpep_pickup_datetime"))
    # janela temporal do arquivo mensal (Jan/2024)
    .filter(col("tpep_pickup_datetime") >= "2024-01-01")
    .filter(col("tpep_pickup_datetime") < "2024-02-01")
    .filter(col("trip_distance") > 0)
    .filter(col("fare_amount") >= 0)
    .filter((col("passenger_count").isNull()) | (col("passenger_count") >= 1))
    .withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))
    .filter(col("pickup_date").isNotNull())
)

In [6]:
from pyspark.sql.functions import min, max

df_silver.select(
    min("tpep_pickup_datetime").alias("min_pickup"),
    max("tpep_pickup_datetime").alias("max_pickup"),
    min("pickup_date").alias("min_pickup_date"),
    max("pickup_date").alias("max_pickup_date"),
).show(truncate=False)

+-------------------+-------------------+---------------+---------------+
|min_pickup         |max_pickup         |min_pickup_date|max_pickup_date|
+-------------------+-------------------+---------------+---------------+
|2024-01-01 00:00:00|2024-01-31 23:59:55|2024-01-01     |2024-01-31     |
+-------------------+-------------------+---------------+---------------+



In [7]:
before = df_bronze.count()
after = df_silver.count()

dropped = before - after
drop_rate = dropped / before if before else 0

print("bronze_rows:", before)
print("silver_rows:", after)
print("dropped_rows:", dropped)
print("drop_rate:", drop_rate)

bronze_rows: 2964624
silver_rows: 2839435
dropped_rows: 125189
drop_rate: 0.042227614699199625


In [8]:
from pyspark.sql.functions import sum, when

diagnostics = df_bronze.select(
    sum(when(col("tpep_pickup_datetime").isNull(), 1).otherwise(0)).alias("null_pickup"),
    sum(when(col("tpep_dropoff_datetime").isNull(), 1).otherwise(0)).alias("null_dropoff"),
    sum(when(col("tpep_dropoff_datetime") < col("tpep_pickup_datetime"), 1).otherwise(0)).alias("invalid_time_order"),
    sum(when(col("trip_distance") <= 0, 1).otherwise(0)).alias("invalid_distance"),
    sum(when(col("fare_amount") < 0, 1).otherwise(0)).alias("invalid_fare"),
    sum(when((col("passenger_count").isNotNull()) & (col("passenger_count") < 1), 1).otherwise(0)).alias("invalid_passenger")
)

diagnostics.show()

+-----------+------------+------------------+----------------+------------+-----------------+
|null_pickup|null_dropoff|invalid_time_order|invalid_distance|invalid_fare|invalid_passenger|
+-----------+------------+------------------+----------------+------------+-----------------+
|          0|           0|                56|           60371|       37448|            31465|
+-----------+------------+------------------+----------------+------------+-----------------+



In [ ]:
# Fallback for kernel resets (keeps notebook runnable even if config wasn't executed)
DATASET = globals().get("DATASET", "nyc_tlc_yellow")
SILVER_PATH = globals().get("SILVER_PATH", f"../lakehouse/silver/{DATASET}")

df_check = spark.read.format("delta").load(SILVER_PATH)
print("silver_rows_check:", df_check.count())
df_check.select("pickup_date").distinct().orderBy("pickup_date").show(10, truncate=False)
df_check.show(5, truncate=False)